# Reading a dnsjax snapshot with NumPy alone

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gokhanyalniz/dnsjax/blob/main/examples/notebooks/read_snapshot_numpy_only.ipynb)

A `dnsjax` snapshot is an ordinary tar archive wrapping a zarr3 store, and
`dnsjax.analysis` reads it with **NumPy and the standard library — no JAX,
no solver runtime**.

This notebook runs a short simulation *as a subprocess* and then
post-processes what it wrote. That is not a stylistic choice: it is what
keeps the demonstration honest, because this kernel never imports the
solver at all. The last cell checks exactly that.

## Setup

Locates the repository, cloning and installing it if this is a fresh
Colab runtime. Nothing here imports `dnsjax`.

In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/gokhanyalniz/dnsjax.git"
EXAMPLE = Path("examples/kolmogorov/parameters.toml")


def locate_example() -> Path:
    """The shipped Kolmogorov example, cloning the repo if need be."""
    for base in (Path.cwd(), *Path.cwd().parents):
        if (base / EXAMPLE).exists():
            return base / EXAMPLE
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "./dnsjax"], check=True
    )
    return (Path("dnsjax") / EXAMPLE).resolve()


example = locate_example()
print("example configuration:", example)

## Run a few steps

The solver runs in its own process, in its own scratch directory, with
the shipped example's `parameters.toml` and one override: stop almost
immediately. Everything else — the flow, the box, the resolution, the
initial condition — comes from the committed file.

In [ ]:
import tempfile

run_dir = Path(tempfile.mkdtemp(prefix="dnsjax-run-"))
shutil.copy(example, run_dir / "parameters.toml")

subprocess.run(
    [
        sys.executable, "-m", "dnsjax",
        "--stop.max_sim_time", "0.5",
        "--outs.it_stats", "10",
        "--outs.snapshot_save_final", "True",
    ],
    cwd=run_dir,
    check=True,
)

snapshots = sorted(run_dir.glob("*.tar"))
print(*(p.name for p in snapshots), sep="\n")

## Read it back — NumPy only

`read_state` pulls the requested data off disk and hands back plain NumPy
arrays, plus the configuration the run was launched with. A triply-periodic
snapshot stores its axes as `(y, z, x)` and its components as
`(u_x, u_y, u_z)`; `geometry_info` reports that schema for any flow, so
post-processing code need not hard-code it.

In [ ]:
from dnsjax.analysis import geometry_info, read_state

data = read_state(snapshots[-1])
u_x, u_y, u_z = data.physical
y, z, x = data.physical_coords

info = geometry_info(data.params)
print("family    ", info.family)
print("components", info.components)
print("shape     ", u_x.shape, u_x.dtype)
print("Re        ", data.params.phys.re)
print("box       ", f"lx={data.params.geo.lx}, lz={data.params.geo.lz}")

## A plane through the spot

The initial condition is a deterministic localized roll, so this is the
same picture on every run.

In [ ]:
import matplotlib.pyplot as plt

mid = u_x.shape[0] // 2          # the y plane through the shear maximum
field = u_x[mid]                 # (z, x)

fig, ax = plt.subplots(figsize=(6, 5), constrained_layout=True)
lim = float(abs(field).max())
im = ax.pcolormesh(
    x, z, field, cmap="RdBu_r", vmin=-lim, vmax=lim, shading="nearest"
)
ax.set_xlabel("x")
ax.set_ylabel("z")
ax.set_title(f"$u_x$ at $y = {y[mid]:.2f}$")
ax.set_aspect("equal")
fig.colorbar(im, ax=ax)
plt.show()

## The solver's own discrete operators

`dnsjax.analysis.snapshot_ops` reproduces the discrete operators the solver
steps with, node for node — not a generic finite difference that happens to
be close. So the divergence of a stepped state comes out at round-off, which
is the property the formulation is built to have, and this is how you check
it from outside the solver.

In [ ]:
import numpy as np

from dnsjax.analysis import divergence, read_state

spec = read_state(snapshots[-1], return_spectral=True, return_physical=False)
div = divergence(spec.spectral, spec.params, spec.spectral_coords)

scale = max(float(np.abs(c).max()) for c in spec.spectral)
print(f"max |div u| / max |u_hat| = {float(np.abs(div).max()) / scale:.3e}")

## The claim, checked

Everything above ran in this kernel. If `dnsjax.analysis` had reached for
JAX, or for the solver runtime, it would be in `sys.modules` by now.

In [ ]:
loaded = sorted(m for m in sys.modules if m.split(".")[0] in
                {"jax", "jaxlib", "zarr", "tensorstore"})
print("JAX / zarr modules imported by this kernel:", loaded or "none")
assert not loaded, loaded

print("dnsjax modules imported by this kernel:")
print(*sorted(m for m in sys.modules if m.startswith("dnsjax")), sep="\n  ")